In [1]:
!pip install -Uq llmcompressor datasets transformers accelerate

In [ ]:
import gc
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

gc.collect()
torch.cuda.empty_cache()

model_path = "/kaggle/input/notebooks/shamsccs/arabicegymedicalmerging/ArEgy-Medical-Command-R7B-Merged"
quant_path = "./ArEgy-Medical-Command-R7B-AWQ-vLLM"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.bfloat16, 
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

import random
import uuid
from datasets import Dataset

system_preamble = """التعليمات:
أنت مساعد طبي دقيق. أجب على سؤال المريض بناءً على الوثائق المسترجعة فقط.
تجاهل الأخطاء الإملائية والكلمات المقلوبة الناتجة عن القراءة الآلية (OCR).
يجب أن تنتهي كل معلومة تكتبها بذكر المصدر حصراً بهذا الشكل: [المصدر: اسم المصدر، صفحة: رقم أو None].
لا تضف أي معلومات من خارج النص."""

qdrant_template = """Score: {score}
Qdrant Point ID: {point_id}
Source: {source} | Page: {page}
Text: {messy_text}"""

raw_scenarios = [
    {
        "source": "دليل السكري العربي", "page": "12",
        "messy_text": "الدم في السكر انخفاض يعني نقص الجلوكوز . اقل من 70 يعتبر منخف9ا . المريض يشعر بـ رعشة ، تع8 ، وتعر8 بارد . ا1علاج السريع هو تناول 15 جرام من سكريات سريعة 3صير أو عسJ . لا تعطي المري9 انسولين ابدا.",
        "question": "أنا حاسس برعشة وعرق ساقع وقست السكر لقيته 60، أعمل إيه بسرعة؟",
        "diagnosis": "تعاني من انخفاض في مستوى السكر بالدم (أقل من 70)، وأعراضه تشمل الرعشة والتعرق البارد.",
        "treatment": "يجب تناول 15 جرام من السكريات السريعة مثل العصير أو العسل فوراً. يُحذر تماماً من أخذ الأنسولين."
    },
    {
        "source": "كتاب أمراض القلب", "page": "88",
        "messy_text": "الدم ضغط ارتفاع صامت قاتل يسمى . الطبيعي الضغط يتراوح 120/80 . اذا زاد عن 140/90 ملم زئ8ق يعتبر مرتفعا . الاعرا9 قد تشمل صداع ، دوار ، وطنيv في الاذن . ا1علاج يشمل تقليJ الملح ، وادوية خافضة للضغط مثJ مثبطات 2CE .",
        "question": "ضغطي 150/95 وعندي زنة في ودني وصداع، هل ده ضغط عالي وإيه علاجه؟",
        "diagnosis": "نعم، تعاني من ارتفاع ضغط الدم حيث أن قراءتك تتجاوز المعدل الطبيعي (140/90)، مما يسبب الصداع وطنين الأذن.",
        "treatment": "العلاج يعتمد على تقليل الملح في الطعام، واستخدام أدوية خافضة للضغط مثل مثبطات ACE."
    },
    {
        "source": "طب الطوارئ", "page": "None",
        "messy_text": "الزائ]ة ال]ودية التهاب حالة طارئة . يبدا الم غام9 حول السرة ينتقJ لاسفل البطv اليمين . غثيان وقيء شائ3 . يمن3 اعطاء مسكنات قوية لانها تخف! الاعرا9 . الت]خل الجراح! لاستئصالها هو الحJ .",
        "question": "عندي وجع غريب بدأ عند السرة ودلوقتي ناقح عليا تحت في الجنب اليمين وعايز أرجع، أخد مسكن قوي؟",
        "diagnosis": "الأعراض تشير إلى التهاب الزائدة الدودية، وهو حالة طوارئ جراحية تبدأ بألم حول السرة ينتقل لأسفل اليمين مع غثيان.",
        "treatment": "يُمنع تماماً أخذ مسكنات قوية لأنها قد تخفي الأعراض. يجب التدخل الجراحي لاستئصالها."
    },
    {
        "source": "المرجع في الباطنة", "page": "104",
        "messy_text": "المع]ة قرحة تسبب الما حار8ا يزدا> بالجوع . ايبوبروفيv ومسكنات N5AIDs تزي] القرحة ونزيفها . العلاج بمث8طات مضخة البروتون ppi مثJ اوميبرازوJ . شرب حلي8 بارد يه]ئ الالم مؤقتا .",
        "question": "أعاني من ألم حارق في فم المعدة يزداد عندما أجوع، هل أستطيع تناول الإيبوبروفين لتسكين الألم؟",
        "diagnosis": "أنت تعاني من قرحة المعدة، والتي تسبب ألماً حارقاً يزداد عند الجوع.",
        "treatment": "يُحذر من تناول الإيبوبروفين والمسكنات لأنها تزيد النزيف والقرحة. العلاج الأساسي هو أوميبرازول (مثبطات مضخة البروتون)، ويمكن شرب حليب بارد للتهدئة."
    },
    {
        "source": "MedlinePlus - IBS", "page": "None",
        "messy_text": "العص8ي القولون متلازمة تسبب مغص وانتفاخ يت8ادل فيه اسهال وامساك . التوتر يزي] الاعراض سوءا . لا يوج] علاج نهائ! لكv يمكن السيطرة عليه بمضادات التقلصات وتجن8 الاطعمة الم7يجة وتقليJ الضغط النفس! .",
        "question": "بطني دايماً منفوخة وفيها مغص، وشوية يجيلي إمساك وشوية إسهال، خصوصاً أيام الامتحانات، ده إيه وعلاجه إيه؟",
        "diagnosis": "هذه أعراض متلازمة القولون العصبي، والتي تسبب مغصاً وانتفاخاً وتبادلاً بين الإسهال والإمساك، وتزداد مع التوتر (أيام الامتحانات).",
        "treatment": "لا يوجد علاج نهائي، ولكن يتم السيطرة عليه بمضادات التقلصات، تجنب الأطعمة المهيجة، وتقليل الضغط النفسي."
    },
    {
        "source": "أمراض الصدر والتنفس", "page": "55",
        "messy_text": "الربو نوبات تضيق شعب هوائية تسبب صفيﺮ بالص]ر وكحة وصعو8ة تنفس . محفزات تشمJ تراب ، دخان ، قطط . ا1علاج السري3 بخاخة سالبوتاموJ موس3 للشع8 . الكورتيزون الوقائ! يستخ]م للحماية طويلة الام] .",
        "question": "صدري بيزيق ومش قادر آخد نفسي بعد ما نظفت التراب اللي في الشقة، إيه الحل السريع؟",
        "diagnosis": "أنت تمر بنوبة ربو ناتجة عن التعرض للتراب (محفز)، مما يسبب ضيق الشعب الهوائية وصعوبة التنفس والتزييق (صفير).",
        "treatment": "العلاج السريع هو استخدام بخاخة سالبوتامول كموسع للشعب الهوائية."
    },
    {
        "source": "المرشد في الكلى", "page": "210",
        "messy_text": "الكلى حصوات تسبب الما شديدا ف! الظهر والجان8 ينتقJ لاسفJ الحو9 . يصاح8ه دم ف! البول . العلاج الم8دئي شرب ماء بكثرة ، مسكنات ، ومضادات تشنج وحاصرات الفا لتسهيJ خروج الحصوة .",
        "question": "يا دكتور عندي وجع فظيع في ضهري من ورا ونازل على الحوض، ولقيت نقط دم في البول، أعمل إيه؟",
        "diagnosis": "الأعراض تشير إلى حصوات الكلى، والتي تسبب ألماً شديداً في الظهر والجانب ويمتد للحوض، مصحوباً بدم في البول.",
        "treatment": "العلاج المبدئي يشمل شرب الماء بكثرة، أخذ مسكنات ومضادات تشنج، وحاصرات ألفا لتسهيل خروج الحصوة."
    },
    {
        "source": "أساسيات أمراض الدم", "page": "41",
        "messy_text": "الدم فقر ( انيميا ) نقw الحدي] هو الاكثر شيوعا . اعراضه تع8 مستمر ، شحو8 الوج7 ، ودوخة عن] الوقوف . العلاج مكملات حدي] تؤخذ م3 فيتاميv سي 1يتحسv الامتصاص . الشاي يمن3 امتصاص الحدي] .",
        "question": "عندي دوخة لما أقف ووشي أصفر ودايماً هبطان، ينفع آخد حديد مع كوباية شاي؟",
        "diagnosis": "أنت تعاني من فقر الدم (أنيميا) الناتج عن نقص الحديد، وأعراضه التعب المستمر، شحوب الوجه، والدوخة عند الوقوف.",
        "treatment": "يجب تناول مكملات الحديد مع فيتامين سي لتحسين الامتصاص. يُحذر تماماً من شرب الشاي معه لأنه يمنع امتصاص الحديد."
    },
    {
        "source": "الطب النبوي والعشبي", "page": "19",
        "messy_text": "النقرس داء الملوك ينتج عن تراكm حم9 اليوريك ف! ال]م . يسبب التهاب حاد والم واحمرار ف! مفصJ الاص8ع الاكبر للقدم . العلاج تقليJ اللحوم الحمراء ، شرب ماء بكثرة ، وادوية مثJ كولشيسيv في النوبة الحادة و الوبيورينوJ للوقاية .",
        "question": "صحيت من النوم صباع رجلي الكبير وارم وأحمر ونار قايدة فيه، ده نقرس؟ وإيه علاجه؟",
        "diagnosis": "نعم، هذا مرض النقرس الناتج عن تراكم حمض اليوريك، ويسبب التهاباً حاداً واحمراراً في مفصل الإصبع الأكبر للقدم.",
        "treatment": "في النوبة الحادة يتم استخدام دواء (كولشيسين). للوقاية، يجب تقليل اللحوم الحمراء، شرب الماء بكثرة، واستخدام (ألوبيورينول)."
    },
    {
        "source": "Healthline - GERD_Ar", "page": "None",
        "messy_text": "المريء ارتجاع GERD يحدث رجوع حم9 المع]ة للمريء . يسبب حرقان بالص]ر ( حرقة المع]ة ) وطعم حام9 بالفm . يزدا> عند الاستلقاء بع] الاكل . ا1علاج مضادات حموضة ، تقليJ وزv ، وع]م النอม بع] الاكل الا بـ 3 ساعات .",
        "question": "ما هو التشخيص لشخص يعاني من حرقان في الصدر وطعم حامض في الفم يزداد عند النوم بعد العشاء؟ وما العلاج؟",
        "diagnosis": "التشخيص هو ارتجاع المريء (GERD)، الناتج عن رجوع حمض المعدة، مما يسبب حرقاناً في الصدر وطعماً حامضاً.",
        "treatment": "العلاج يشمل مضادات الحموضة، تقليل الوزن، ويُمنع النوم بعد الأكل إلا بعد مرور 3 ساعات."
    },
    {
        "source": "الطب النفسي والعصبي", "page": "115",
        "messy_text": "النصف! الصداع شقيقة تسبب الما نابضا ف! جان8 واح] مv الراس . قد يصاح8ه غثيان وحساسية للضوء والصوت . تري8تان Triptans ه! الادوية الافضJ لوقف النوبة . يج8 الجلوس ف! غرفة مظلمة وهادئة .",
        "question": "نص دماغي اليمين بتنط من الوجع ومش طايق أشوف نور ولا أسمع صوت، إيه العلاج؟",
        "diagnosis": "أنت تعاني من نوبة صداع نصفي (شقيقة)، وتسبب ألماً نابضاً في جانب واحد من الرأس مع حساسية للضوء والصوت.",
        "treatment": "يجب الجلوس في غرفة مظلمة وهادئة، والأدوية الأفضل لوقف النوبة هي أدوية التريبتان (Triptans)."
    },
    {
        "source": "مكافحة العدوى والسموم", "page": "203",
        "messy_text": "الغذائ! التسمm ينتج عن طعام ملوث . اسهال مائ! ، قيء ، ومغw . اخطر مضاعفاته الجفاف . ا1علاج الاساسي محاليJ الجفاف 0RS . يمن3 منعا باتا اعطاء دواء لوبيرامي] L0peramide لانه يح8س البكتيريا ف! الامعاء .",
        "question": "أكلت من برة وبطني بتتقطع وإسهال وترجيع، ينفع آخد دواء لوبيراميد عشان أوقف الإسهال؟",
        "diagnosis": "تعاني من تسمم غذائي نتيجة طعام ملوث، ويسبب الإسهال المائي والقيء والمغص. الخطر الأكبر هو الجفاف.",
        "treatment": "العلاج الأساسي هو محاليل معالجة الجفاف (ORS). يُمنع منعاً باتاً أخذ دواء لوبيراميد لأنه يحبس البكتيريا في الأمعاء."
    },
    {
        "source": "دليل الغدد الصماء", "page": "74",
        "messy_text": "ال]رقية الغدة خمول يسبب بطء الايض . المري9 يشتك! مv تع8 ، زيا>ة وزv رغm قل4 الاكل ، حساسيه لل8رد ، وتساقط شعﺮ . العلاج تعويضي بهرمون ثيروكسيv Lev0thyroxine يؤخذ على مع]ة فارغة صباحا .",
        "question": "وزني بيزيد لوحده وبحس ببرد شديد وشعري بيقع ودايماً تعبانة، ده خمول غدة؟ وباخد العلاج إمتى؟",
        "diagnosis": "نعم، الأعراض تشير إلى خمول في الغدة الدرقية، مما يسبب التعب، زيادة الوزن، الحساسية للبرد، وتساقط الشعر.",
        "treatment": "العلاج يتم بتعويض الهرمون عبر دواء (ليفوثيروكسين)، ويجب أن يؤخذ على معدة فارغة في الصباح."
    },
    {
        "source": "جراحة العظام والمفاصل", "page": "310",
        "messy_text": "الرك8ة خشونة تاكل الغضاريف . الم يزدا> م3 صعود السلm والوقوف طويلا . العلاج تمرينات لتقوية عضلات الفخ] ، مسكنات موض3ية ، وحقv حم9 الهيالورونيk ف! المفصJ . تخفي! الوزv ضروري ج]ا .",
        "question": "ما هو علاج ألم الركبة الذي يزداد بشدة عند صعود السلالم؟",
        "diagnosis": "أنت تعاني من خشونة الركبة الناتجة عن تآكل الغضاريف، والتي تزداد ألمها مع صعود السلالم.",
        "treatment": "يجب تخفيف الوزن، ممارسة تمارين لتقوية عضلات الفخذ، واستخدام مسكنات موضعية أو حقن حمض الهيالورونيك."
    },
    {
        "source": "طب الأطفال العملي", "page": "22",
        "messy_text": "الاطفاJ حرارة ارتفاع يتطل8 مراق8ة . اذا كانت فنز8 38.5 اعط! باراسيتاموJ او ايبوبروفيv . يمن3 الاس8رين تماما للاطفاJ لانه قد يسبب متلازمة راي Reye fatal . كمادات ماء فاتر وليw بارد او ثلج .",
        "question": "ابني حرارته 38.5، ينفع أديله أسبرين وأعمله كمادات تلج؟",
        "diagnosis": "يعاني طفلك من ارتفاع في درجة الحرارة ويحتاج لمراقبة وتخفيض للحرارة.",
        "treatment": "يمكن إعطاء باراسيتامول أو إيبوبروفين مع كمادات ماء فاتر. يُمنع تماماً استخدام الأسبرين للأطفال (يسبب متلازمة راي)، ويُمنع استخدام الثلج للكمادات."
    },
]

extra_scenarios = [
    {**raw_scenarios[0], "question": "ما هي الإسعافات الأولية لغيبوبة انخفاض السكر إذا كانت القراءة 60؟"},
    {**raw_scenarios[1], "question": "هل يعتبر ضغط 150/95 مرتفعاً؟ وما هو بروتوكول العلاج؟"},
    {**raw_scenarios[2], "question": "ما هي الأعراض الكلاسيكية لالتهاب الزائدة الدودية ولماذا تمنع المسكنات؟"},
    {**raw_scenarios[3], "question": "وجع فم المعدة اللي بيزيد مع الجوع، إيه علاجه السريع؟"},
    {**raw_scenarios[4], "question": "ما هو علاج القولون العصبي المرتبط بالتوتر النفسي؟"},
    {**raw_scenarios[5], "question": "ما هو العلاج الدوائي المباشر لنوبة الربو الحادة الناتجة عن الغبار؟"},
    {**raw_scenarios[6], "question": "ما هي الإجراءات الطبية المبدئية للتعامل مع حصوات الكلى ونزيف البول؟"},
    {**raw_scenarios[7], "question": "هل يؤثر الشاي على مكملات الحديد لعلاج فقر الدم؟"},
    {**raw_scenarios[8], "question": "ما هو داء الملوك وما هي طرق الوقاية منه طبياً؟"},
    {**raw_scenarios[11], "question": "ما هو العلاج الأساسي للتسمم الغذائي وما هو الدواء الممنوع استخدامه؟"}
]

raw_scenarios.extend(extra_scenarios)

synthetic_data = []

for i in range(64):
    scenario = raw_scenarios[i % len(raw_scenarios)]
    
    point_id = str(uuid.uuid4())
    score = round(random.uniform(0.65, 0.95), 4)
    
    docs = qdrant_template.format(
        score=score, 
        point_id=point_id, 
        source=scenario['source'], 
        page=scenario['page'], 
        messy_text=scenario['messy_text']
    )
    
    target_answer = f"- التشخيص: {scenario['diagnosis']} [المصدر: {scenario['source']}، صفحة: {scenario['page']}]\n- العلاج: {scenario['treatment']} [المصدر: {scenario['source']}، صفحة: {scenario['page']}]"
    
    full_prompt = f"{system_preamble}\n\nالوثائق المسترجعة:\n{docs}\n\nسؤال المريض: {scenario['question']}\n\nالإجابة المختصرة والموثقة:\n{target_answer}"
    
    synthetic_data.append({"text": full_prompt})

calib_dataset = Dataset.from_list(synthetic_data)

recipe = [
    AWQModifier(
        targets=["Linear"], 
        ignore=["lm_head"], 
        scheme="W8A16" 
    )
]

print("The GPUs are using bfloat16 to process global attention without NaNs...")

oneshot(
    model=model,
    dataset=calib_dataset,
    recipe=recipe,
    
    max_seq_length=1024,          
    num_calibration_samples=64, 
    batch_size=2          
)

model.save_pretrained(quant_path)
tokenizer.save_pretrained(quant_path)

print("AWQ Quantization complete, officially vLLM-ready.")

2026-03-17 15:26:20.608696: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773761180.630942     169 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773761180.638092     169 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773761180.656352     169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773761180.656371     169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773761180.656374     169 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Synthesizing 25+ Realistic Qdrant RAG Examples with Heavy OCR Noise...
✅ Successfully loaded 64 hardcore RAG calibration examples!
The GPUs are using bfloat16 to process global attention without NaNs...
2026-03-17T15:27:03.975819+0000 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


Tokenizing:   0%|          | 0/64 [00:00<?, ? examples/s]

2026-03-17T15:27:05.779114+0000 | _make_sampler | WARNING - Shuffling the dataset can lead to unoptimal batching for sequence lengths non-uniform sizes. When collating with truncation, this will delete a large number of tokens. When collating with padding, this will add a large number of padding tokens.

Please consider calling `oneshot` with `batch_size=1`
2026-03-17T15:27:05.780428+0000 | reset | INFO - Compression lifecycle reset
2026-03-17T15:27:05.782895+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-03-17T15:27:05.836812+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-03-17T15:27:05.849919+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-03-17T15:27:05.850825+0000 | _set_resolved_mappings | WARNING - skipping AWQ for

W0317 15:27:05.933000 169 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Grid search for model.layers.0.input_layernorm:  50%|█████     | 10/20 [02:14<02:13, 13.35s/it, best_error=1.431e-06]